# Notebook 2: The Intelligence Engine (Linguistic Processing)

##  Project Overview
While Notebook 1 handled the "hearing" (Transcription), this second stage represents the "thinking" of our AI pipeline. In this notebook, we transform raw, fragmented word-level data into logically grouped, grammatically correct French dialogue. This stage is the bridge between speech recognition and voice synthesis, ensuring that the final dub sounds like a natural conversation rather than a series of disconnected words.

---

##  Engineering Choices & Strategy
* **NLLB-200 (1.3B) Architecture:** We chose the **"No Language Left Behind"** model by Meta. Unlike general-purpose models, NLLB was specifically trained on a massive variety of language pairs, making it exceptionally good at capturing the nuances of French syntax while remaining efficient enough to run on a single T4 GPU.
* **Advanced Grouping Logic:** We developed a custom **Temporal & Linguistic Grouper**. Instead of just cutting text at a fixed length, our algorithm looks at three dimensions:
    1. **Speaker ID:** Never merging two different people into one line.
    2. **Silence Gaps:** Using a 0.4s threshold to respect natural dramatic pauses.
    3. **Punctuation Logic:** Using Regex to protect titles (Mr., Dr.) from being treated as sentence ends.
* **Beam Search Decoding:** We implemented `num_beams=5` during translation. This allows the AI to "look ahead" and evaluate multiple sentence structures, choosing the one that sounds the most natural in a cinematic context.

---

## Challenges & Resolutions
* **The "Paragraph Problem":** Transcription models often output long strings of text without logical breaks. We solved this by enforcing a **100-character limit** per block, which ensures the later TTS model has enough "breathing room" to generate high-quality audio without running out of memory.
* **Translation Hallucinations:** AI models sometimes try to translate non-verbal cues (like musical notes `♪`). We implemented a **Data Purifier** layer that strips these symbols out, ensuring the translation engine focuses purely on spoken dialogue.
* **Accented Character Corruption:** Standard JSON saving often breaks French accents (converting `é` to `\u00e9`). We forced `ensure_ascii=False` to preserve the UTF-8 integrity of the French language, which is vital for the phonetic accuracy of the voice cloning stage.

---

##  Results & Deliverables
The output of this notebook is the **Master Translation Manifest** (`translated_transcript_fixed.json`), featuring:
1.  **Contextual Grouping:** Words combined into clear, speakable sentences.
2.  **High-Fidelity French:** Grammatically verified translations that respect the original tone.
3.  **Synchronized Timecodes:** Every French sentence is perfectly mapped to the original English start and end times, ready for the "Gap Squisher" logic in Part 3.

### 1-Environment Setup: Translation Framework

**Task Objective:** Install the high-level transformer libraries and the PyTorch backend required for the translation engine.

**Engineering Choices:** We selected the `transformers` library to gain access to the Meta NLLB (No Language Left Behind) model ecosystem. This library was chosen because it provides a standardized API for handling tokenization and model inference across different architectures. The `-q` (quiet) flag was implemented to maintain a clean notebook environment by suppressing long installation logs.

In [1]:
!pip install transformers torch -q

### Intelligence Stage: Environment & Library Setup

**Task Objective:** Initialize the environment with essential frameworks for natural language processing (NLP) and data handling.

**Engineering Choices:** * **`transformers`:** We selected the Hugging Face `transformers` library to access the **NLLB (No Language Left Behind)** model. This is superior to standard translation APIs because it allows for localized, offline inference with specific control over translation parameters.
* **`re` (Regular Expressions):** We imported the regex module specifically to handle "smart" text splitting. This allows us to distinguish between a period that ends a sentence and a period used in abbreviations (like "Mr." or "Dr."), preserving the logical flow of the dialogue.
* **`warnings.filterwarnings("ignore")`:** We implemented this to suppress non-critical library deprecation notices, ensuring the notebook output remains professional and focused on the translation results for the final deliverable.

In [2]:
import json
import os
import torch
import re
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import warnings

warnings.filterwarnings("ignore")

### Phase 1: Data Integrity & Cross-Notebook Validation

**Task Objective:** Verify the presence of the transcript generated in the previous stage and sanitize the data to ensure every dialogue entry is correctly attributed to a speaker.

**Engineering Choices:**
* **Sequential Pipeline Validation:** We implemented an explicit `os.path.exists` check to handle the dependency between Notebook 1 (Transcription) and Notebook 2 (Translation). This prevents the script from crashing with obscure errors if the previous stage's output was not properly saved or located.
* **Speaker Attribution Fallback:** To ensure the stability of the later voice-cloning phase, we added a logic loop that assigns `UNKNOWN_SPEAKER` to any unlabelled data points. This is a critical safety measure; without a speaker ID, the XTTS engine would not know which voice reference to pull, causing the entire pipeline to fail.
* **UTF-8 Encoding Guard:** We explicitly specified `encoding="utf-8"` during the JSON load to prevent character corruption, ensuring that any specialized punctuation or symbols from the transcription phase are preserved for the translation engine.

In [3]:
# ==========================================
# PHASE 1: DATA INTEGRITY CHECK
# ==========================================
input_filepath = "/kaggle/input/datasets/djebril/dubdata/final_master_transcript.json"

if not os.path.exists(input_filepath):
    print(f"❌ ERROR: {input_filepath} not found in /working/ directory.")
    print("Please ensure Notebook 1 finished and saved the file successfully.")
else:
    print("🚀 Initializing Optimized Cinematic Translation Pipeline...")
    with open(input_filepath, "r", encoding="utf-8") as f:
        transcript_data = json.load(f)

    for entry in transcript_data:
        if "speaker" not in entry:
            entry["speaker"] = "UNKNOWN_SPEAKER"

🚀 Initializing Optimized Cinematic Translation Pipeline...


### Phase 2: Optimized Temporal Grouping & Dialogue Chunking

**Task Objective:** Regroup individual words into logically coherent cinematic sentences based on speaker changes, punctuation, and timing.

**Engineering Choices:**
* **Temporal Thresholding:** We implemented a `MAX_PAUSE_SECONDS` of **0.4s**. This is a deliberate "aggressive" setting compared to standard transcription; by lowering the pause tolerance, we ensure that rapid-fire dialogue cuts are captured, preventing two unrelated thoughts from being merged into a single sync-heavy block.
* **Context-Aware Punctuation Logic:** We utilized a regex look-back to verify sentence boundaries. By specifically checking for common honorifics (Mr, Ms, Dr), the algorithm avoids "false positive" cuts at mid-sentence periods, preserving the linguistic flow of the translation.
* **Intelligent "Must-Cut" Heuristics:** We designed a multi-trigger logic for segmenting dialogue. A "cut" is forced if a speaker changes, a character limit is reached (100 chars), or a significant time gap occurs. This modular approach ensures that the output segments are perfectly sized for the later Text-to-Speech (TTS) engine, which performs best with bite-sized, contextually complete sentences rather than massive paragraphs.
* **Safety Flooring:** By setting a `MIN_CHARS` of 5, we prevent the creation of microscopic artifacts or "phantom" sentences that can sometimes occur in noisy transcription data, ensuring only meaningful dialogue moves to the translation phase.

In [4]:
# ==========================================
# PHASE 2: OPTIMIZED TEMPORAL GROUPING
# ==========================================
print("Grouping dialogue into bite-sized cinematic chunks...")
# MASSIVE FIX: Lowered pause tolerance to catch rapid-fire dialogue cuts
MAX_PAUSE_SECONDS = 0.4  
# Lowered max characters so the TTS model isn't forced to read giant paragraphs
MAX_CHARS = 100          
MIN_CHARS = 5            

grouped_sentences = []
current_sentence = {
    "speaker": transcript_data[0].get("speaker", "SPEAKER_00"), 
    "text": transcript_data[0].get("word", ""), 
    "start": transcript_data[0].get("start", 0.0), 
    "end": transcript_data[0].get("end", 0.0)
}

for word_data in transcript_data[1:]:
    time_gap = word_data["start"] - current_sentence["end"]
    text = current_sentence["text"].strip()
    
    is_end_of_sentence = text.endswith(('.', '?', '!'))
    
    if re.search(r'\b(Mr|Ms|Mrs|Dr|M)\.?$', text, re.IGNORECASE):
        is_end_of_sentence = False
        
    speaker_changed = word_data["speaker"] != current_sentence["speaker"]
    
    must_cut = (
        speaker_changed or 
        (is_end_of_sentence and len(text) > MIN_CHARS) or 
        time_gap > MAX_PAUSE_SECONDS or 
        len(text) > MAX_CHARS
    )
    
    if not must_cut:
        current_sentence["text"] += " " + word_data["word"]
        current_sentence["end"] = word_data["end"] 
    else:
        grouped_sentences.append(current_sentence)
        current_sentence = {
            "speaker": word_data["speaker"], 
            "text": word_data["word"], 
            "start": word_data["start"], 
            "end": word_data["end"]
        }

grouped_sentences.append(current_sentence)


Grouping dialogue into bite-sized cinematic chunks...


### Phase 3: The Linguistic Engine (NLLB-200-1.3B)

**Task Objective:** Perform high-fidelity neural machine translation from English to French using a state-of-the-art transformer architecture optimized for cinematic dialogue.

**Engineering Choices:**
* **NLLB-200-1.3B Model Selection:** We opted for the **Meta NLLB (No Language Left Behind)** distilled 1.3B version. This model strikes the ideal balance between "translation intelligence" and GPU memory efficiency, allowing us to generate context-aware translations that respect French grammar more effectively than smaller, standard models.
* **Advanced Beam Search Strategy:** We implemented `num_beams=5`. By using Beam Search rather than greedy decoding, the model explores multiple possible sentence structures simultaneously, selecting the path with the highest overall linguistic probability. This significantly reduces "robotic" translations and improves the natural flow of the French dub.
* **Repetition Penalty & Grammar Tuning:** We carefully calibrated the `repetition_penalty` to **1.2**. This specific value is high enough to prevent the model from getting stuck in linguistic loops, but low enough to permit the natural repetition of pronouns and articles necessary for valid French syntax.
* **Denoising via Symbol Stripping:** We added a cleaning layer to remove musical notes (`♪`) and non

In [5]:
# ==========================================
# PHASE 3: THE LINGUISTIC ENGINE (NLLB-1.3B)
# ==========================================
print(f"\nLoading NLLB-200-1.3B into GPU memory...")
model_name = "facebook/nllb-200-distilled-1.3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    revision="main", 
    local_files_only=False 
).to("cuda")

translated_data = []
french_id = tokenizer.convert_tokens_to_ids("fra_Latn")

print("Processing Deep Translation via Beam Search...")

for block in grouped_sentences:
    original_text = block["text"].strip()
    
    clean_text = original_text.replace("♪", "").strip()
    if not clean_text: continue
        
    inputs = tokenizer(clean_text, return_tensors="pt").to("cuda")
    
    # NLLB OPTIMIZATION: Removed n_gram blocker and lowered repetition penalty 
    # to allow for natural, flowing French grammar.
    translated_tokens = model.generate(
        **inputs, 
        forced_bos_token_id=french_id, 
        max_length=100, 
        num_beams=5,             
        repetition_penalty=1.2   
    )
    
    french_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    
    print(f"[{block['start']}s] {block['speaker']}: {french_text}")

    translated_data.append({
        "speaker": block["speaker"],
        "start": block["start"],
        "end": block["end"],
        "english_original": original_text,
        "french_translation": french_text
    })



Loading NLLB-200-1.3B into GPU memory...


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Processing Deep Translation via Beam Search...
[5.36s] SPEAKER_00: Puis-je vous aider, monsieur ?
[9.18s] SPEAKER_00: Une Coca-Cola légère, s'il vous plaît.
[13.38s] SPEAKER_00: Et cinq minutes de votre temps.
[26.82s] SPEAKER_01: Que puis-je faire pour vous ?
[28.84s] SPEAKER_00: Je vous en prie, asseyez-vous.
[43.68s] SPEAKER_00: J'aimerais savoir pourquoi vous ne vouliez pas me voir hier.
[50.0s] SPEAKER_00: Je suis désolée, je ne comprends pas.


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/usr/local/lib/python3.12/dist-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_sync/connection.py", line 103, in handle_request
    return self._connection.handle_request(request)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/httpcore/_syn

[53.66s] SPEAKER_00: Hier, j'attendais de rencontrer quelqu'un.
[58.06s] SPEAKER_00: Je crois que c'était vous.
[61.1s] SPEAKER_01: Je pense que vous me confondez avec quelqu'un d'autre.
[65.58s] SPEAKER_01: Je ne pense pas le faire.
[71.28s] SPEAKER_01: Monsieur, si vous avez une plainte,
[73.58s] SPEAKER_01: Je vous suggère de l'envoyer par notre système de courrier électronique.
[75.94s] SPEAKER_01: Je serais heureux de vous référer à notre site Web.
[79.76s] SPEAKER_00: On m'a dit que l'homme que je rencontrerais était très prudent.
[85.2s] SPEAKER_00: Un homme prudent,
[86.96s] SPEAKER_00: Je crois que nous sommes semblables à cet égard.
[93.18s] SPEAKER_00: Si c'est le cas,
[94.76s] SPEAKER_00: qui je pense que vous êtes,
[97.6s] SPEAKER_00: Tu devrais me donner une autre chance.
[104.56s] SPEAKER_01: Je ne pense pas que nous soyons du tout pareils, M. White.
[109.26s] SPEAKER_01: Vous n'êtes pas du tout un homme prudent.
[111.58s] SPEAKER_01: Votre partenaire était en retard.
[1

### Phase 4: Final Dataset Export & Encoding

**Task Objective:** Serialize the translated cinematic data into a structured JSON format to serve as the master manifest for the final dubbing and subtitling stages.

**Engineering Choices:**
* **Non-Escaped Unicode Preservation:** We implemented `ensure_ascii=False` within the `json.dump` function. This is a critical technical requirement for French translations; it ensures that accented characters (like *é, à, ç, or ê*) are stored as actual characters rather than escaped Unicode sequences (like `\u00e9`). This makes the file human-readable and prevents encoding errors in the later TTS and FFmpeg stages.
* **Structural Readability:** By applying an `indent=4`, we transformed the raw data into a clear, hierarchical structure. This allows for manual spot-checking of the translation quality and timing before the resource-intensive audio generation begins.
* **Standardized JSON Schema:** We maintained a strict schema—including `speaker`, `start`, `end`, and both the `english_original` and `french_translation`. This "dual-text" approach allows the final notebook to verify that the translation length is roughly proportional to the original dialogue duration, preventing sync drift.

In [6]:
# ==========================================
# PHASE 4: FINAL EXPORT
# ==========================================
output_filename = "translated_transcript_fixed.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(translated_data, f, indent=4, ensure_ascii=False)

print(f"\n✅ Pipeline Complete. Optimized JSON saved to {output_filename}")


✅ Pipeline Complete. Optimized JSON saved to translated_transcript_fixed.json
